# Traspaso de Blacklist CIM → MGRS

Este notebook extrae los blacklists de tiles CIM desde `params.tsv` y los traspasa a tiles MGRS, fusionando las escenas descartadas de todos los tiles CIM que intersectan cada tile MGRS.

**Flujo:**
1. Lee `params.tsv` (blacklists por tile CIM, año, satélite)
2. Consulta GEE para encontrar qué tiles CIM intersectan cada tile MGRS
3. Fusiona los blacklists por (tile MGRS, año, satélite)
4. Escribe `input_params_mgrs.json` listo para el script de mosaico

## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo si no tienes earthengine-api instalado
# !pip install earthengine-api

## 1. Imports

In [ ]:
import csv
import json
from collections import defaultdict
from pathlib import Path

import ee

## 2. Autenticación y configuración GEE

La primera vez que ejecutes esto necesitas autenticarte. Después queda guardado.

In [ ]:
# Solo necesitas ejecutar esto UNA VEZ para autenticarte.
# Abre el link que aparece, copia el código y pégalo aquí.
# ee.Authenticate()

In [ ]:
GEE_PROJECT = "mapbiomas-chile"

ee.Initialize(project=GEE_PROJECT)
print("GEE inicializado correctamente")

## 3. Parámetros de configuración

Ajusta las rutas y tiles según tu entorno.

In [ ]:
# ── Rutas de archivos ────────────────────────────────────────────────────────
TSV_PATH    = Path("params.tsv")               # Ruta a tu params.tsv
OUTPUT_PATH = Path("input_params_mgrs.json")   # JSON de salida

# ── Assets GEE ──────────────────────────────────────────────────────────────
CIM_GRID_ASSET  = "projects/mapbiomas-workspace/AUXILIAR/cim-world-1-250000"
MGRS_GRID_ASSET = "projects/sat-io/open-datasets/MGRS/MGRS_100km"

# ── Tiles MGRS de prueba para Chile ─────────────────────────────────────────
MGRS_TILES_CONFIG = {
    "18FXH": {"crs": "EPSG:32718"},
    "18GXP": {"crs": "EPSG:32718"},
    "18HYD": {"crs": "EPSG:32718"},
    "19HCD": {"crs": "EPSG:32719"},
    "19JCJ": {"crs": "EPSG:32719"},
    "19KDU": {"crs": "EPSG:32719"},
}

# ── Filtros opcionales ───────────────────────────────────────────────────────
# Dejar en None para procesar todos
YEARS_FILTER = None        # Ejemplo: [2018, 2019, 2020]
TILES_FILTER = None        # Ejemplo: ["19JCJ", "18HYD"]

print("Configuración cargada")
print(f"  TSV:    {TSV_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Tiles MGRS: {list(MGRS_TILES_CONFIG.keys())}")

## 4. Lectura del params.tsv

In [ ]:
def parse_blacklist(value):
    """Parsea el campo BLACK LIST del TSV a lista de strings."""
    if not value:
        return []
    return [x.strip() for x in value.replace(";", ",").split(",") if x.strip()]


def parse_bool(value, default=True):
    if not value:
        return default
    return str(value).strip().lower() not in {"false", "0", "no", "n", "off"}


def load_tsv(tsv_path):
    """Lee params.tsv y devuelve lista de registros."""
    records = []
    with open(tsv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            year_str  = (row.get("YEAR") or "").strip()
            grid_name = (row.get("GRID_NAME") or "").strip()
            satellite = (row.get("SATELLITE") or "").strip().lower()
            if satellite and not satellite.startswith("l"):
                satellite = f"l{satellite}"
            black_list    = parse_blacklist(row.get("BLACK LIST"))
            use_tile_mask = parse_bool(row.get("USETILEMASK"), default=True)

            if not year_str or not grid_name or not satellite:
                continue

            records.append({
                "year":          int(year_str),
                "grid_name":     grid_name,
                "satellite":     satellite,
                "black_list":    black_list,
                "use_tile_mask": use_tile_mask,
            })
    return records

In [ ]:
tsv_records = load_tsv(TSV_PATH)

print(f"Filas cargadas:          {len(tsv_records)}")
print(f"Filas con blacklist:     {sum(1 for r in tsv_records if r['black_list'])}")
print(f"Tiles CIM únicos:        {len({r['grid_name'] for r in tsv_records})}")
print(f"Años cubiertos:          {min(r['year'] for r in tsv_records)} – {max(r['year'] for r in tsv_records)}")
print(f"Satélites:               {sorted({r['satellite'] for r in tsv_records})}")

## 5. Consulta GEE: qué tiles CIM intersectan cada tile MGRS

In [ ]:
def get_cim_tiles_for_mgrs(mgrs_tile_names, cim_name_field="name", mgrs_name_field="name"):
    """
    Para cada tile MGRS devuelve la lista de nombres de tiles CIM que lo intersectan.
    
    Returns:
        dict: {mgrs_tile_name: [cim_tile_name, ...]}
    """
    mgrs_fc = ee.FeatureCollection(MGRS_GRID_ASSET)
    cim_fc  = ee.FeatureCollection(CIM_GRID_ASSET)
    result  = {}

    for mgrs_name in mgrs_tile_names:
        print(f"  Consultando: {mgrs_name}...", end=" ")

        mgrs_geom = mgrs_fc \
            .filter(ee.Filter.eq(mgrs_name_field, mgrs_name)) \
            .first() \
            .geometry()

        cim_names = cim_fc \
            .filterBounds(mgrs_geom) \
            .reduceColumns(ee.Reducer.toList(), [cim_name_field]) \
            .get("list") \
            .getInfo()

        result[mgrs_name] = sorted(cim_names) if cim_names else []
        print(f"→ {len(result[mgrs_name])} tiles CIM")

    return result

In [ ]:
tiles_to_process = (
    [t for t in TILES_FILTER if t in MGRS_TILES_CONFIG]
    if TILES_FILTER
    else list(MGRS_TILES_CONFIG.keys())
)

print("Consultando intersecciones CIM ↔ MGRS en GEE...")
mgrs_to_cim = get_cim_tiles_for_mgrs(tiles_to_process)

print("\nResultado:")
for mgrs, cims in mgrs_to_cim.items():
    print(f"  {mgrs}: {cims}")

## 6. Fusión de blacklists CIM → MGRS

In [ ]:
def build_mgrs_blacklist_index(tsv_records, mgrs_to_cim):
    """
    Construye índice: (mgrs_tile, year, satellite) → [scene_ids]
    Fusiona blacklists de todos los tiles CIM que intersectan el tile MGRS.
    """
    # Índice TSV: (cim_tile, year, satellite) → [scene_ids]
    cim_index = defaultdict(list)
    for rec in tsv_records:
        key = (rec["grid_name"], rec["year"], rec["satellite"])
        cim_index[key].extend(rec["black_list"])

    # Todos los pares (year, satellite) del TSV
    year_sat_pairs = sorted({(rec["year"], rec["satellite"]) for rec in tsv_records})

    mgrs_index = {}
    for mgrs_name, cim_tiles in mgrs_to_cim.items():
        for year, satellite in year_sat_pairs:
            merged = []
            for cim_tile in cim_tiles:
                merged.extend(cim_index.get((cim_tile, year, satellite), []))

            # Eliminar duplicados manteniendo orden
            seen, unique = set(), []
            for s in merged:
                if s not in seen:
                    seen.add(s)
                    unique.append(s)

            mgrs_index[(mgrs_name, year, satellite)] = unique

    return mgrs_index, year_sat_pairs

In [ ]:
mgrs_index, year_sat_pairs = build_mgrs_blacklist_index(tsv_records, mgrs_to_cim)

# Mostrar resumen de blacklists fusionados
print("Blacklists fusionados por tile MGRS:\n")
for mgrs_name in tiles_to_process:
    entries_with_bl = [
        (year, sat, scenes)
        for (tile, year, sat), scenes in mgrs_index.items()
        if tile == mgrs_name and scenes
    ]
    total = sum(len(s) for _, _, s in entries_with_bl)
    print(f"  {mgrs_name}: {len(entries_with_bl)} años con blacklist, {total} escenas en total")
    for year, sat, scenes in sorted(entries_with_bl):
        print(f"    {year} ({sat.upper()}): {len(scenes)} escenas")

## 7. Generación del JSON de salida

In [ ]:
def get_satellite_for_year(year, tsv_records):
    """Devuelve el satélite más frecuente en el TSV para un año dado."""
    counts = defaultdict(int)
    for rec in tsv_records:
        if rec["year"] == year:
            counts[rec["satellite"]] += 1
    if counts:
        return max(counts, key=counts.__getitem__)
    # Fallback histórico
    if year <= 1999:   return "l5"
    elif year <= 2002: return "l7"
    elif year == 2003: return "l5"
    elif year <= 2012: return "l7"
    elif year <= 2022: return "l8"
    else:              return "l9"


def build_output_records(mgrs_index, tsv_records, mgrs_tiles_config,
                         tiles_to_process, years_filter=None):
    all_years = sorted({rec["year"] for rec in tsv_records})
    if years_filter:
        all_years = [y for y in all_years if y in years_filter]

    output = []
    for mgrs_name in tiles_to_process:
        for year in all_years:
            satellite = get_satellite_for_year(year, tsv_records)
            black_list = mgrs_index.get((mgrs_name, year, satellite), [])

            output.append({
                "country":       "CHILE",
                "grid_name":     mgrs_name,
                "year":          year,
                "satellite":     satellite,
                "t0_s":          f"{year}-01-01",
                "t1_s":          f"{year}-12-31",
                "cloud_cover":   80,
                "black_list":    black_list,
                "use_tile_mask": False,
                "crs":           mgrs_tiles_config[mgrs_name]["crs"],
                "grid_type":     "MGRS",
            })
    return output

In [ ]:
output_records = build_output_records(
    mgrs_index=mgrs_index,
    tsv_records=tsv_records,
    mgrs_tiles_config=MGRS_TILES_CONFIG,
    tiles_to_process=tiles_to_process,
    years_filter=YEARS_FILTER,
)

print(f"Total registros generados: {len(output_records)}")
print(f"Registros con blacklist:   {sum(1 for r in output_records if r['black_list'])}")
print(f"Total escenas blacklist:   {sum(len(r['black_list']) for r in output_records)}")

## 8. Vista previa del JSON

In [ ]:
# Mostrar los primeros registros con blacklist no vacío
ejemplos = [r for r in output_records if r["black_list"]][:3]

if ejemplos:
    print("Ejemplos de registros con blacklist:\n")
    print(json.dumps(ejemplos, indent=2, ensure_ascii=False))
else:
    print("No hay registros con blacklist para los tiles/años seleccionados.")
    print("Primer registro:")
    print(json.dumps(output_records[0], indent=2, ensure_ascii=False))

## 9. Guardar JSON

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output_records, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"✓ Archivo guardado: {OUTPUT_PATH}")
print(f"  Tamaño: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

## 10. Reporte final

In [ ]:
print("=" * 60)
print("REPORTE FINAL: TRASPASO CIM → MGRS")
print("=" * 60)

for mgrs_name in tiles_to_process:
    cim_tiles = mgrs_to_cim.get(mgrs_name, [])
    tile_records = [r for r in output_records if r["grid_name"] == mgrs_name]
    con_bl = [r for r in tile_records if r["black_list"]]
    total_escenas = sum(len(r["black_list"]) for r in tile_records)

    print(f"\nTile MGRS : {mgrs_name} ({MGRS_TILES_CONFIG[mgrs_name]['crs']})")
    print(f"CIM tiles : {len(cim_tiles)} → {cim_tiles}")
    print(f"Años      : {len(tile_records)}")
    print(f"Con BL    : {len(con_bl)} años")
    print(f"Escenas BL: {total_escenas}")

print("\n" + "=" * 60)
print(f"JSON listo para usar: {OUTPUT_PATH}")
print("Próximo paso: ejecutar el script de mosaico con este JSON")